In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib as mpl
from matplotlib.lines import Line2D

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm

from adjustText import adjust_text

The raw data for the material stock analysis was provided in different units—some reported per dwelling, others per building. To ensure consistency and comparability, the data was converted into a common unit. Two approaches were applied:

## Approach 1: Converting dwelling-level data into disaggregated building-level data

In this approach, the number of dwellings was converted to the building level, after which the material stock was assessed.

In [2]:
# ---------------------------
# SSB 03175 – Building-Level Data Cleaner
# ---------------------------
class SSB03175DataCleaner:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_and_clean_data(self):
        df = pd.read_excel(self.file_path, header=None)
        df = df.drop(index=range(4))  
        df = df.drop(index=range(366, 561), errors='ignore') 
        df.iloc[0, 0] = "kommunenum"
        df.columns = df.iloc[0]
        df = df.drop(df.index[0]).reset_index(drop=True)
        df = df.dropna(how='all')
        exclude_munis = [
            'K-21-22 Svalbard and Jan Mayen',
            'K-23 Continental shelf',
            'K-Rest Divided municalities and unknown'
        ]
        df = df[~df['kommunenum'].isin(exclude_munis)]
        return df

# ---------------------------
# SSB 06266 – Dwelling-Level Data Cleaner
# ---------------------------
class SSB06266DataCleaner:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_and_clean_data(self):
        df = pd.read_excel(self.file_path, header=None)
        df.iloc[4, 1] = "kommunenum"
        df = df.drop(df.index[:4])
        df.columns = df.iloc[0]
        df = df.drop(df.index[0]).reset_index(drop=True)
        df = df.drop(df.columns[0], axis=1)
        df = df.dropna(how='all')
        exclude_vals = [
            'K-21-22 Svalbard and Jan Mayen',
            'K-23 Continental shelf',
            'K-Rest Divided municalities and unknown'
        ]
        df = df[~df['kommunenum'].isin(exclude_vals)]
        self.clean_column_names(df)
        return df

    def clean_column_names(self, df):
        df.columns = df.columns.str.replace("2021 and after", "2021", regex=False)
        df.columns = df.columns.str.replace('-', '_')
        df.columns.values[1:14] = ['Detached house_' + col for col in df.columns[1:14]]
        df.columns.values[14:27] = ['House with 2 dwellings_' + col for col in df.columns[14:27]]
        df.columns.values[27:40] = ['Row house, linked house and house with 3 dwellings or more_' + col
                                    for col in df.columns[27:40]]
        df.columns.values[40:53] = ['Multi-dwelling building_' + col for col in df.columns[40:53]]
        df.columns.values[53:] = ['Residence for communities_' + col for col in df.columns[53:]]
        return df

# ---------------------------
# Calculator: Sum Dwelling Counts by Archetype
# ---------------------------
class BuildingDwellingCalculator:
    def __init__(self, dwellings_df):
        self.data = dwellings_df

    def calculate_sums_by_category(self):
        summary = pd.DataFrame()
        summary['kommunenum'] = self.data['kommunenum'].values
        summary['Detached house'] = self.data.filter(like='Detached house_').sum(axis=1)
        summary['House with 2 dwellings'] = self.data.filter(like='House with 2 dwellings_').sum(axis=1)
        summary['Row house, linked house and house with 3 dwellings or more'] = \
            self.data.filter(like='Row house, linked house and house with 3 dwellings or more_').sum(axis=1)
        summary['Multi-dwelling building'] = self.data.filter(like='Multi-dwelling building_').sum(axis=1)
        summary['Residence for communities'] = self.data.filter(like='Residence for communities_').sum(axis=1)
        return summary

# ---------------------------
# Calculator: Dwelling Count per Building
# ---------------------------
class DwellingsPerBuildingCalculator:
    def __init__(self, dwellings_total, buildings_df):
        self.dwellings_total = dwellings_total
        self.buildings_data = buildings_df

    def calculate_dwellings_per_building(self):
        ratio_df = pd.DataFrame()
        ratio_df['kommunenum'] = self.dwellings_total['kommunenum'].values
        buildings_nonzero = self.buildings_data.replace(0, pd.NA)
        for category in self.dwellings_total.columns[1:]:
            ratio_df[category] = self.dwellings_total[category] / buildings_nonzero[category]
        return ratio_df

# ---------------------------
# Calculator: Disaggregated Building Data
# ---------------------------
class BuildingsDisaggregatedCalculator:
    def __init__(self, dwelling_data, dwellings_ratio):
        self.dwelling_data = dwelling_data
        self.dwellings_ratio = dwellings_ratio

    def calculate_disaggregated_buildings(self):
        disaggregated = self.dwelling_data[['kommunenum']].copy()
        for archetype in self.dwellings_ratio.columns[1:]:
            matching_columns = self.dwelling_data.filter(like=archetype).columns
            if not matching_columns.empty:
                for col in matching_columns:
                    denominator = self.dwellings_ratio[archetype]
                    disaggregated[col] = self.dwelling_data[col] / denominator.where(denominator != 0, 1)
        disaggregated = disaggregated.replace([float('inf'), -float('inf')], pd.NA).fillna(0)
        return disaggregated

# ---------------------------
# Function: Transform Building Data Columns
# ---------------------------
def transform_buildings_disaggregated(df):
    key = "kommunenum"
    filtered_cols = [col for col in df.columns if col == key or "Unknown" not in col]
    df = df[filtered_cols]
    # Rename archetypes for clarity.
    df.rename(columns=lambda col: col.replace("Detached house", "SFH")
              .replace("House with 2 dwellings", "MFH")
              .replace("Row house, linked house and house with 3 dwellings or more", "MFH")
              .replace("Multi-dwelling building", "AB")
              .replace("Residence for communities", "AB"), inplace=True)
    kommunenum_series = df[key]
    df_values = df.drop(columns=key)
    # Group the numeric columns by their archetype (i.e. sum columns sharing the same archetype)
    df_values = df_values.groupby(level=0, axis=1).sum()
    df_values.insert(0, key, kommunenum_series)
    return df_values

# ---------------------------
# Function: Rename Year Ranges
# ---------------------------
def rename_year_ranges(df):
    df.rename(
        columns=lambda col: (col if col == "kommunenum"
                             else col.replace("1946_1960", "1955")
                                     .replace("1941_1945", "1955")
                                     .replace("1921_1940", "1955")
                                     .replace("1901_1920", "1955")
                                     .replace("1900 and earlier", "1955")
                                     .replace("1961_1970", "1956_1970")),
        inplace=True
    )
    return df

# ---------------------------
# Function: Process Material Inventory Data
# ---------------------------
def process_material_inventory(file_path):
    mi_df = pd.read_excel(file_path)
    mi_df.drop(["Unnamed: 0", "Unit"], axis=1, inplace=True)
    archetypes_2011_2020 = mi_df[mi_df['Archetype'].str.endswith('2011_2020')]
    archetypes_2021 = archetypes_2011_2020.copy()
    archetypes_2021['Archetype'] = archetypes_2021['Archetype'].str.replace('2011_2020', '2021')
    mi_df = pd.concat([mi_df, archetypes_2021]).reset_index(drop=True)
    return mi_df

# ---------------------------
# Municipalities Converter
# ---------------------------
class MunicipalitiesConverter:
    def __init__(self, data):
        self.data = data

    def clean_municipalities(self):
        self.data = self.data[~self.data['kommunenum'].isin([
            'K-21-22 Svalbard and Jan Mayen',
            'K-23 Continental shelf',
            'K-Rest Divided municalities and unknown'
        ])]
        self.data['municipality_code'] = self.data['kommunenum'].str.slice(start=2, stop=6)
        self.data = self.data.drop(columns=['kommunenum'])
        self.data = self.data.rename(columns={'municipality_code': 'kommunenum'})
        self.data = self.data.set_index('kommunenum')
        return self.data

# ---------------------------
# Helper Function: Format Final Column Names
# ---------------------------
def format_output_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

# ---------------------------
# Main Orchestration
# ---------------------------
def main_approach1():
    outputs = {}  

    # File paths
    ssb03175_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/ssb_03175/ssb_03175_2024_raw.xlsx"
    ssb06266_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/ssb_06266/ssb_06266_2024_full.xlsx"
    material_inventory_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/mi_no_updated.xlsx"

    # ---------------------------
    # Load and clean building-level data (SSB 03175)
    # ---------------------------
    building_cleaner = SSB03175DataCleaner(ssb03175_file)
    cleaned_buildings_df = building_cleaner.load_and_clean_data()
    outputs["cleaned_buildings_df"] = cleaned_buildings_df.copy()
    
    # ---------------------------
    # Load and clean dwelling-level data (SSB 06266)
    # ---------------------------
    dwellings_cleaner = SSB06266DataCleaner(ssb06266_file)
    dwellings_df = dwellings_cleaner.load_and_clean_data()
    outputs["dwellings_df"] = dwellings_df.copy()

    # ---------------------------
    # Calculate summed dwelling counts by archetype
    # ---------------------------
    dwelling_calculator = BuildingDwellingCalculator(dwellings_df)
    dwellings_total = dwelling_calculator.calculate_sums_by_category()
    outputs["dwellings_total"] = dwellings_total.copy()

    # ---------------------------
    # Calculate dwellings per building (ratio)
    # ---------------------------
    ratio_calculator = DwellingsPerBuildingCalculator(dwellings_total, cleaned_buildings_df)
    dwellings_ratio = ratio_calculator.calculate_dwellings_per_building()
    outputs["dwellings_ratio"] = dwellings_ratio.copy()

    # ---------------------------
    # Disaggregate building-level data
    # ---------------------------
    disaggregator = BuildingsDisaggregatedCalculator(dwellings_df, dwellings_ratio)
    buildings_disaggregated = disaggregator.calculate_disaggregated_buildings()
    outputs["buildings_disaggregated"] = buildings_disaggregated.copy()

    # ---------------------------
    # Transform and rename building data columns
    # ---------------------------
    buildings_transformed = transform_buildings_disaggregated(buildings_disaggregated)
    buildings_transformed = rename_year_ranges(buildings_transformed)
    outputs["buildings_transformed"] = buildings_transformed.copy()

    # ---------------------------
    # Standardize municipality identifiers
    # ---------------------------
    municipality_converter = MunicipalitiesConverter(buildings_transformed)
    buildings_with_municipalities = municipality_converter.clean_municipalities()
    outputs["buildings_with_municipalities"] = buildings_with_municipalities.copy()

    # ---------------------------
    # Melt the building data from wide to long format
    # ---------------------------
    buildings_melted = buildings_with_municipalities.reset_index().melt(
        id_vars='kommunenum', var_name='Archetype', value_name='building_count'
    )
    outputs["buildings_melted"] = buildings_melted.copy()

    # Group the melted data by municipality and archetype (summing building counts).
    buildings_melted_grouped = buildings_melted.groupby(['kommunenum', 'Archetype'], as_index=False)['building_count'].sum()
    outputs["buildings_melted_grouped"] = buildings_melted_grouped.copy()

    # ---------------------------
    # Process material inventory data.
    # ---------------------------
    material_inventory_data = process_material_inventory(material_inventory_file)
    outputs["material_inventory_data"] = material_inventory_data.copy()

    # ---------------------------
    # Merge building counts with material inventory data & calculate material stocks.
    # ---------------------------
    buildings_mi_merged = pd.merge(buildings_melted_grouped, material_inventory_data, on='Archetype')
    buildings_mi_merged['total_material_amounts'] = buildings_mi_merged['building_count'] * buildings_mi_merged['Value']
    buildings_mi_merged = format_output_columns(buildings_mi_merged)
    outputs["buildings_mi_merged"] = buildings_mi_merged.copy()


    # Save the final output.
    outputs["final_output"] = buildings_mi_merged.copy()
    return outputs

if __name__ == "__main__":
    
    results_approach1 = main_approach1()

    # Create separate DataFrames from the dictionary.
    cleaned_buildings_df      = results_approach1["cleaned_buildings_df"]
    dwellings_df              = results_approach1["dwellings_df"]
    dwellings_total           = results_approach1["dwellings_total"]
    dwellings_ratio           = results_approach1["dwellings_ratio"]
    buildings_disaggregated   = results_approach1["buildings_disaggregated"]
    buildings_transformed     = results_approach1["buildings_transformed"]
    buildings_with_municipalities = results_approach1["buildings_with_municipalities"]
    buildings_melted          = results_approach1["buildings_melted"]
    buildings_melted_grouped  = results_approach1["buildings_melted_grouped"]
    material_inventory_df     = results_approach1["material_inventory_data"]
    buildings_mi_merged       = results_approach1["buildings_mi_merged"]
    mat_stock_approach1    = results_approach1["final_output"]


/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/3350705328.py:120: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns=lambda col: col.replace("Detached house", "SFH")
/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/3350705328.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_values = df_values.groupby(level=0, axis=1).sum()


## Approach 2: Estimating material stock based on total heated floor area at the municipal level

In this approach, the heated floor area per dwelling was used to calculate the total heated floor area for each municipality. The resulting values were then multiplied by material intensity factors to estimate the material stock.

In [3]:
# ---------------------------
# SSB06266 – Dwelling-Level Data Cleaner
# ---------------------------
class SSB06266DataCleaner:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def load_and_clean_data(self):
        df = pd.read_excel(self.file_path, header=None)
        df.iloc[4, 1] = "kommunenum"
        df = df.drop(df.index[:4])
        df.columns = df.iloc[0]
        df = df.drop(df.index[0]).reset_index(drop=True)
        df = df.drop(df.columns[0], axis=1)
        df = df.dropna(how='all')
        exclude_vals = [
            'K-21-22 Svalbard and Jan Mayen',
            'K-23 Continental shelf',
            'K-Rest Divided municalities and unknown'
        ]
        df = df[~df['kommunenum'].isin(exclude_vals)]
        self.clean_column_names(df)
        return df

    def clean_column_names(self, df):
        df.columns = df.columns.str.replace("2021 and after", "2021", regex=False)
        df.columns = df.columns.str.replace('-', '_')
        df.columns.values[1:14] = ['Detached house_' + col for col in df.columns[1:14]]
        df.columns.values[14:27] = ['House with 2 dwellings_' + col for col in df.columns[14:27]]
        df.columns.values[27:40] = ['Row house, linked house and house with 3 dwellings or more_' + col 
                                    for col in df.columns[27:40]]
        df.columns.values[40:53] = ['Multi-dwelling building_' + col for col in df.columns[40:53]]
        df.columns.values[53:] = ['Residence for communities_' + col for col in df.columns[53:]]
        return df

# ---------------------------
# Process Heated Floor Data
# ---------------------------
def process_heated_floor_data(file_path):
    df = pd.read_excel(file_path)
    df = df.set_index(df.columns[0])
    df.reset_index(inplace=True)
    melted = df.melt(id_vars=df.columns[0], var_name="Year", value_name="Value")
    melted["Combined"] = melted[df.columns[0]] + "_" + melted["Year"]
    pivoted_df = melted.pivot_table(index=None, columns="Combined", values="Value")
    pivoted_df.reset_index(drop=True, inplace=True)
    pivoted_df.rename(
        columns=lambda col: col.replace("_1800", "1955")
                           .replace("1801_1955", "1955")
                           .replace("2021_2050", "2021"),
        inplace=True
    )
    pivoted_df = pivoted_df.groupby(axis=1, level=0).mean()
    return pivoted_df

# ---------------------------
# Prepare Dwelling Data
# ---------------------------
def prepare_dwelling_data(dw_df):
    exclude_substrings = ["Unknown"]
    filtered_columns = [col for col in dw_df.columns if not any(sub in col for sub in exclude_substrings)]
    dw_df = dw_df[filtered_columns]

    dw_df.rename(columns=lambda col: col.replace("Detached house", "SFH")
                 .replace("House with 2 dwellings", "TH")
                 .replace("Row house, linked house and house with 3 dwellings or more", "TH")
                 .replace("Multi-dwelling building", "MFH")
                 .replace("Residence for communities", "MFH"), inplace=True)
    
    dw_df = dw_df.groupby(axis=1, level=0).sum()

    dw_df.rename(columns=lambda col: col.replace("1946_1960", "1955")
                 .replace("1941_1945", "1955")
                 .replace("1921_1940", "1955")
                 .replace("1901_1920", "1955")
                 .replace("1900 and earlier", "1955")
                 .replace("1961_1970", "1956_1970"), inplace=True)
    
    dw_df = dw_df.groupby(axis=1, level=0).sum()
    return dw_df

# ---------------------------
# Compute Total Heated Floor Area
# ---------------------------
def compute_heated_floor_area(dw_df, floor_area_df):
    av_broadcasted = pd.concat([floor_area_df] * len(dw_df), ignore_index=True)
    dw_numeric = dw_df.loc[:, dw_df.columns != 'kommunenum']
    av_numeric = av_broadcasted.loc[:, av_broadcasted.columns != 'Combined']
    
    dw_numeric = dw_numeric[av_numeric.columns]
    result_numeric = dw_numeric * av_numeric
    total_heated_area = pd.concat([dw_df[['kommunenum']], result_numeric], axis=1)

    total_heated_area.rename(columns=lambda col: col.replace("MFH", "AB").replace("TH", "MFH"), inplace=True)
    return total_heated_area

# ---------------------------
# Process Material Inventory Data
# ---------------------------
def process_material_inventory(file_path):
    mi_df = pd.read_excel(file_path)
    mi_df.drop(["Unnamed: 0", "Unit"], axis=1, inplace=True)
    archetypes_2011_2020 = mi_df[mi_df['Archetype'].str.endswith('2011_2020')]
    archetypes_2021 = archetypes_2011_2020.copy()
    archetypes_2021['Archetype'] = archetypes_2021['Archetype'].str.replace('2011_2020', '2021')
    mi_df = pd.concat([mi_df, archetypes_2021]).reset_index(drop=True)
    mi_df.rename(columns={"archetype": "Archetype"}, inplace=True)
    return mi_df

# ---------------------------
# Process Archetype Material Data
# ---------------------------
def process_archetype_material_data(archetype_area_file, material_inventory_df):
    archetype_df = pd.read_excel(archetype_area_file)

    
    if 'DB_Area' not in archetype_df.columns:
        possible = [col for col in archetype_df.columns if 'area' in col.lower()]
        if possible:
            archetype_df.rename(columns={possible[0]: 'DB_Area'}, inplace=True)
        else:
            raise KeyError("No 'DB_Area' column found or similar column to infer.")

    archetypes_2011_2020 = archetype_df[archetype_df['Archetype'].str.endswith('2011_2020')]
    archetypes_2021 = archetypes_2011_2020.copy()
    archetypes_2021['Archetype'] = archetypes_2021['Archetype'].str.replace('2011_2020', '2021')
    archetype_df = pd.concat([archetype_df, archetypes_2021]).reset_index(drop=True)
    
    merged_df = pd.merge(archetype_df, material_inventory_df, on='Archetype', how='inner')
    merged_df['Material_intensity'] = merged_df['Value'] / merged_df['DB_Area']
    return merged_df, archetype_df

# ---------------------------
# Helper Function
# ---------------------------
def format_output_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

# ---------------------------
# Municipalities Converter
# ---------------------------
class MunicipalitiesConverter:
    def __init__(self, data):
        self.data = data

    def clean_municipalities(self):
        self.data = self.data[~self.data['kommunenum'].isin([
            'K-21-22 Svalbard and Jan Mayen',
            'K-23 Continental shelf',
            'K-Rest Divided municalities and unknown'
        ])]
        self.data['municipality_code'] = self.data['kommunenum'].str.slice(start=2, stop=6)
        self.data = self.data.drop(columns=['kommunenum'])
        self.data = self.data.rename(columns={'municipality_code': 'kommunenum'})
        self.data = self.data.set_index('kommunenum')
        return self.data

# ---------------------------
# Main Orchestration
# ---------------------------
def main_approach2():
    outputs = {}

    ssb06266_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/ssb_06266/ssb_06266_2024_full.xlsx"
    heated_floor_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/Sandberg_2017_Table_B1.xlsx"
    archetype_area_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/archetypes_areas.xlsx"
    material_inventory_file = "/Users/dolgayamaria/Thesis/Practical Part/Data/mi_no_updated.xlsx"

    dwellings_cleaner = SSB06266DataCleaner(ssb06266_file)
    dwellings_df = dwellings_cleaner.load_and_clean_data()
    outputs["dwellings_df"] = dwellings_df.copy()

    dwellings_prepared = prepare_dwelling_data(dwellings_df)
    outputs["dwellings_prepared"] = dwellings_prepared.copy()

    heated_floor_df = process_heated_floor_data(heated_floor_file)
    outputs["heated_floor_df"] = heated_floor_df.copy()

    total_heated_area = compute_heated_floor_area(dwellings_prepared, heated_floor_df)
    outputs["total_heated_area"] = total_heated_area.copy()

    municipality_converter = MunicipalitiesConverter(total_heated_area)
    total_heated_area_cleaned = municipality_converter.clean_municipalities()
    outputs["total_heated_area_cleaned"] = total_heated_area_cleaned.copy()

    total_heated_area_melted = total_heated_area_cleaned.reset_index().melt(
        id_vars='kommunenum', var_name='Archetype', value_name='Total Heated Area'
    )
    outputs["total_heated_area_melted"] = total_heated_area_melted.copy()

    material_inventory_data = process_material_inventory(material_inventory_file)
    outputs["material_inventory_data"] = material_inventory_data.copy()

    material_intensity_df, archetype_area_df = process_archetype_material_data(archetype_area_file, material_inventory_data)
    outputs["material_intensity_data"] = material_intensity_df.copy()
    outputs["archetype_area_data"] = archetype_area_df.copy()

    heated_area_mi_merged = pd.merge(
        total_heated_area_melted, material_intensity_df, on='Archetype'
    )
    heated_area_mi_merged['Total Material Amounts'] = (
        heated_area_mi_merged['Material_intensity'] * heated_area_mi_merged['Total Heated Area']
    )
    outputs["heated_area_mi_merged"] = heated_area_mi_merged.copy()

    heated_area_mi_merged = format_output_columns(heated_area_mi_merged)
    outputs["final_output"] = heated_area_mi_merged.copy()

    return outputs

# ---------------------------
# Execute Script
# ---------------------------
if __name__ == "__main__":
    results_approach2 = main_approach2()
    keys_list = [
        "dwellings_df", "dwellings_prepared", "heated_floor_df", "total_heated_area",
        "total_heated_area_cleaned", "total_heated_area_melted", "material_inventory_data",
        "archetype_area_data", "material_intensity_data", "heated_area_mi_merged", "final_output"
    ]
    dataset_list = [results_approach2[k] for k in keys_list]
    for name, df in zip(keys_list, dataset_list):
        print(f"--- {name} ---")
        print(df.head(), "\n")

    dwellings_df = results_approach2["dwellings_df"]
    dwellings_prepared = results_approach2["dwellings_prepared"]
    heated_floor_df = results_approach2["heated_floor_df"]
    total_heated_area = results_approach2["total_heated_area"]
    total_heated_area_cleaned = results_approach2["total_heated_area_cleaned"]
    total_heated_area_melted = results_approach2["total_heated_area_melted"]
    material_inventory_data = results_approach2["material_inventory_data"]
    archetype_area_df = results_approach2["archetype_area_data"]
    material_intensity_df = results_approach2["material_intensity_data"]
    heated_area_mi_merged = results_approach2["heated_area_mi_merged"]
    mat_stock_approach_2 = results_approach2["final_output"]


--- dwellings_df ---
4          kommunenum Detached house_1900 and earlier  \
0       K-3101 Halden                             599   
1         K-3103 Moss                             528   
2    K-3105 Sarpsborg                             761   
3  K-3107 Fredrikstad                            2171   
4       K-3110 Hvaler                             270   

4 Detached house_1901_1920 Detached house_1921_1940 Detached house_1941_1945  \
0                      420                      717                       27   
1                      392                      711                       36   
2                      785                     1202                       74   
3                     1242                     1409                       80   
4                      102                      100                        6   

4 Detached house_1946_1960 Detached house_1961_1970 Detached house_1971_1980  \
0                      906                     1119                     118

/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/2210517620.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dw_df.rename(columns=lambda col: col.replace("Detached house", "SFH")
/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/2210517620.py:71: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dw_df = dw_df.groupby(axis=1, level=0).sum()
/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/2210517620.py:80: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dw_df = dw_df.groupby(axis=1, level=0).sum()
/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/2210517620.py:54: FutureWarning: DataFrame.groupby with axis=1 is depreca

Both approaches produced reasonable results; however, Approach 2 was ultimately chosen as it provides a more consistent basis for analysis, aligns better with available data, and allows for direct scaling from heated floor area at the municipal level.

## "Other" dwellings

The raw data from Table 06266 (SSB Statistics Norway, 2024) included an “Other” category of dwellings. It was necessary to assess whether the share of this category was significant compared to the total number of dwellings. Based on this evaluation, a decision was made on whether to retain it in the analysis (with assumptions) or exclude it as negligible.

In [4]:
# ------------------------------
# SSB Other Data Cleaner Class
# ------------------------------
class SSBOtherDataCleaner:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def load_and_clean_data(self):
        self.data = pd.read_excel(self.file_path, header=None)
        self.data = self.data.drop(index=range(4))
        self.data = self.data.drop(index=range(366, 567), errors='ignore')
        self.data = self.data.drop(columns=self.data.columns[0])
        self.data.iloc[0, 0] = "kommunenum"
        self.data.columns = self.data.iloc[0]
        self.data = self.data.drop(self.data.index[0]).reset_index(drop=True)
        self.data = self.data.dropna(how='all')
        self.data.columns = self.data.columns.str.replace('-', '_')
        self.data = self.data[~self.data['kommunenum'].isin([
            'K-21-22 Svalbard and Jan Mayen', 
            'K-23 Continental shelf', 
            'K-Rest Divided municalities and unknown'
        ])]
        self.data['municipality_code'] = self.data['kommunenum'].str.slice(start=2, stop=6)
        self.data = self.data.drop(columns=['kommunenum']).rename(columns={'municipality_code': 'kommunenum'})
        self.data = self.data.set_index('kommunenum')
        return self.data

# ------------------------------
# STEP 1: Load & Clean "OTHER" Data
# ------------------------------
ssb_other_cleaner = SSBOtherDataCleaner("/Users/dolgayamaria/Thesis/Practical Part/Data/ssb_06266/06266_other.xlsx")
other_dwellings = ssb_other_cleaner.load_and_clean_data()

other_dwellings.rename(
    columns=lambda col: col.replace("1946_1960", "Other_1955")
                        .replace("1961_1970", "Other_1956_1970")
                        .replace("1971_1980", "Other_1971_1980")
                        .replace("1981_1990", "Other_1981_1990")
                        .replace("1991_2000", "Other_1991_2000")
                        .replace("2001_2010", "Other_2001_2010")
                        .replace("2011_2020", "Other_2011_2020")
                        .replace("2021 and after", "Other_2021"),
    inplace=True
)
other_dwellings.rename(
    columns=lambda col: col.replace("1941_1945", "Other_1955")
                        .replace("1921_1940", "Other_1955")
                        .replace("1901_1920", "Other_1955")
                        .replace("1900 and earlier", "Other_1955"),
    inplace=True
)
other_dwellings = other_dwellings.groupby(axis=1, level=0).sum()

# ------------------------------
# STEP 2: Load Spatial & Population Data
# ------------------------------
municipal_masks_2024 = gpd.read_file('/Users/dolgayamaria/Thesis/Practical Part/Data/ssb_municipalities_masks/Kommuner 2024.shp')
population_2024 = pd.read_excel(
    "/Users/dolgayamaria/Thesis/Practical Part/Data/01222_20250626-164638.xlsx",
    skiprows=3,
    header=0,
    nrows=360
)

population_2024 = population_2024.rename(columns={population_2024.columns[0]: 'raw'})
exclude = [
    'K-21-22 Svalbard and Jan Mayen',
    'K-23 Continental shelf',
    'K-Rest Divided municalities and unknown'
]
population_2024 = population_2024[~population_2024['raw'].isin(exclude)]

population_2024[['kommunenum', 'kommunenav']] = population_2024['raw'] \
    .str.extract(r'K-(\d{4})\s+(.+)$')

population_2024 = population_2024.drop(columns=['raw'])
population_2024 = population_2024.rename(columns={population_2024.columns[0]: 'population'})
population_2024 = population_2024[['kommunenum', 'kommunenav', 'population']]
municipalities_with_population = municipal_masks_2024.merge(population_2024, on="kommunenum", how="left")
pop_other = municipalities_with_population.merge(other_dwellings, on="kommunenum", how="inner")

# ------------------------------
# STEP 3: Merge with Dwelling Data via MunicipalitiesConverter
# ------------------------------
municipality_converter = MunicipalitiesConverter(dwellings_prepared)
dw_converted = municipality_converter.clean_municipalities()
dw_full_pop = pop_other.merge(dw_converted, on="kommunenum", how="inner")
dw_full_pop.rename(
    columns=lambda col: col.replace("MFH", "AB").replace("TH", "MFH"),
    inplace=True
)

# ------------------------------
# STEP 4: Compute Per Capita Values & Determine Closest Building Type
# ------------------------------
dw_full_pop["Other_per_capita"] = dw_full_pop[[col for col in dw_full_pop.columns if col.startswith("Other_")]].sum(axis=1) / dw_full_pop["population"]
dw_full_pop["SFH_per_capita"] = dw_full_pop[[col for col in dw_full_pop.columns if col.startswith("SFH_")]].sum(axis=1) / dw_full_pop["population"]
dw_full_pop["MFH_per_capita"] = dw_full_pop[[col for col in dw_full_pop.columns if col.startswith("MFH_")]].sum(axis=1) / dw_full_pop["population"]
dw_full_pop["AB_per_capita"] = dw_full_pop[[col for col in dw_full_pop.columns if col.startswith("AB_")]].sum(axis=1) / dw_full_pop["population"]
dw_full_pop["Closest_to"] = dw_full_pop[["SFH_per_capita", "MFH_per_capita", "AB_per_capita"]].sub(dw_full_pop["Other_per_capita"], axis=0).abs().idxmin(axis=1)
dw_other_incl = dw_full_pop.copy()

# ------------------------------
# STEP 5: Allocate "Other" Values to Building Types
# ------------------------------
for btype in ["AB", "SFH", "MFH"]:
    closest_rows = dw_other_incl["Closest_to"] == f"{btype}_per_capita"
    other_cols = [col for col in dw_other_incl.columns if col.startswith("Other_") and not col.endswith("_per_capita")]
    target_cols = [col for col in dw_other_incl.columns if col.startswith(f"{btype}_") and not col.endswith("_per_capita")]
    other_sorted = sorted(other_cols, key=lambda x: int(x.split("_")[1]))
    target_sorted = sorted(target_cols, key=lambda x: int(x.split("_")[1]))
    assert len(target_sorted) == len(other_sorted), f"Mismatch for {btype}"
    for t, o in zip(target_sorted, other_sorted):
        dw_other_incl.loc[closest_rows, t] += dw_other_incl.loc[closest_rows, o]
dw_other_incl.drop(columns=other_cols, inplace=True)
dw_other_incl = dw_other_incl.drop(['geometry', 'population', 'Unknown',
                                      'Other_per_capita', 'SFH_per_capita', 'MFH_per_capita',
                                      'AB_per_capita', 'Closest_to'], axis=1)

# ------------------------------
# STEP 6: Multiply by Average Heated Floor Area
# ------------------------------
heated_floor_df.rename(
    columns=lambda col: col.replace("MFH", "AB").replace("TH", "MFH"),
    inplace=True
)
av_broadcasted = pd.concat([heated_floor_df] * len(dw_other_incl), ignore_index=True)
dw_numeric = dw_other_incl.loc[:, dw_other_incl.columns != 'kommunenum']
av_numeric = av_broadcasted.loc[:, av_broadcasted.columns != 'Combined']
dw_numeric = dw_numeric[av_numeric.columns]
result_numeric = dw_numeric * av_numeric
total_heated_area = pd.concat([dw_other_incl[['kommunenum']], result_numeric], axis=1)
total_heated_area.set_index('kommunenum', inplace=True)
total_heated_area_melted = total_heated_area.reset_index().melt(id_vars='kommunenum', var_name='archetype', value_name='total_heated_area')

# ------------------------------
# STEP 7: Merge with Material Intensity Data & Compute Totals
# ------------------------------
total_heated_area_melted.columns = total_heated_area_melted.columns.str.lower().str.replace(' ', '_')
material_intensity_df.columns = material_intensity_df.columns.str.lower().str.replace(' ', '_')
heated_area_other_mi_merged = total_heated_area_melted.merge(material_intensity_df, on='archetype')
heated_area_other_mi_merged['total_material_amounts'] = heated_area_other_mi_merged['material_intensity'] * heated_area_other_mi_merged['total_heated_area']
heated_area_other_mi_merged.columns = heated_area_other_mi_merged.columns.str.lower().str.replace(' ', '_')

/var/folders/kn/87q7q5h140ldb97ht3jdcpr00000gn/T/ipykernel_99006/2686432927.py:53: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  other_dwellings = other_dwellings.groupby(axis=1, level=0).sum()


In [5]:
# Load datasets
old_data = mat_stock_approach_2.copy() # Without "other" dwellings
new_data = heated_area_other_mi_merged.copy()  # With "other" dwellings

# Summarize total material amounts
total_old = old_data.loc[old_data['material_type'] == 'TOTAL', 'total_material_amounts'].sum()
total_new = new_data.loc[new_data['material_type'] == 'TOTAL', 'total_material_amounts'].sum()

total_diff = total_new - total_old
percent_change = (total_diff / total_old) * 100

print(f"Total Material Amounts (Old): {total_old:.2f} kg")
print(f"Total Material Amounts (New): {total_new:.2f} kg")
print(f"Absolute Difference: {total_diff:.2f} kg, {total_diff / 1_000_000_000:.2f} Mt")
print(f"Percentage Change: {percent_change:.2f}%\n")

Total Material Amounts (Old): 258561924495.03 kg
Total Material Amounts (New): 264040295401.49 kg
Absolute Difference: 5478370906.46 kg, 5.48 Mt
Percentage Change: 2.12%



The “Other” category of dwellings accounted for only 2.12% of the total stock. Since this share is relatively small, the category was excluded from further analysis.

## Monte Carlo simulations

Since the raw data did not include uncertainty ranges, Monte Carlo simulations were applied to derive them. Most of the datasets used in the analysis were simulated with the same number of iterations to ensure consistency. Detailed information on the simulation parameters can be found in the main text of this study.

### 1. Material inventory data

In [26]:
# ----------------------------
# 1. Data Preparation
# ----------------------------
np.random.seed(42)  

material_inventory_data = material_inventory_df.rename(columns={'Value': 'average'})
material_inventory_data = material_inventory_data[material_inventory_data['Material type'] != 'other']
material_inventory_data[['type', 'cohort']] = material_inventory_data['Archetype'].str.split('_', n=1, expand=True)

print("DataFrame after splitting 'Archetype' into 'type' and 'cohort':")
print(material_inventory_data)
print("-" * 60)

# ----------------------------
# 2. Monte Carlo Simulation (Vectorized per Group)
# ----------------------------
num_simulations = 500  


unique_types    = np.sort(material_inventory_data['type'].unique())
unique_cohorts  = np.sort(material_inventory_data['cohort'].unique())
unique_materials = np.sort(material_inventory_data['Material type'].unique())

n_types    = len(unique_types)
n_cohorts  = len(unique_cohorts)
n_materials = len(unique_materials)

type_to_index     = {t: i for i, t in enumerate(unique_types)}
cohort_to_index   = {c: i for i, c in enumerate(unique_cohorts)}
material_to_index = {m: i for i, m in enumerate(unique_materials)}

# Pre-allocate the final 4-D array for simulation results (dimensions: type, cohort, material, sim_run)
material_inventory_array = np.full((n_types, n_cohorts, n_materials, num_simulations), np.nan, dtype=float)

# Group the DataFrame by (type, cohort)
grouped = material_inventory_data.groupby(['type', 'cohort'])
for (grp_type, grp_cohort), group_df in grouped:
    i = type_to_index[grp_type]
    j = cohort_to_index[grp_cohort]
    
    # Get the TOTAL row (if present) which gives the "original total" value.
    total_row = group_df[group_df['Material type'] == 'TOTAL']
    if not total_row.empty:
        tot_orig = total_row['average'].values[0]
    else:
        # If no TOTAL row exists, use the sum of all rows in the group.
        tot_orig = group_df['average'].sum()
        
    # Select the proper simulation range based on the cohort.  
    # For "1955": 20% range (lower = tot_orig*0.8, upper = tot_orig*1.2).
    # For "1956_1970": 15% range (lower = tot_orig*0.85, upper = tot_orig*1.15).
    # For all others: 10% range (lower = tot_orig*0.9, upper = tot_orig*1.1).
    if grp_cohort == '1955':
        lower = tot_orig * 0.7
        upper = tot_orig * 1.3
    elif grp_cohort == '1956_1970':
        lower = tot_orig * 0.8
        upper = tot_orig * 1.2
    else:
        lower = tot_orig * 0.9
        upper = tot_orig * 1.1
        
    mode  = tot_orig
    # Generate simulated total values (vectorized for num_simulations)
    simulated_totals = np.random.triangular(lower, mode, upper, num_simulations) 
    
    # For non-TOTAL materials, we compute the allocation by share.
    non_total_df = group_df[group_df['Material type'] != 'TOTAL']
    if not non_total_df.empty:
        # Compute shares for each material in this group.
        shares = non_total_df['average'].values.astype(float) / tot_orig  
        sim_values = np.outer(shares, simulated_totals)
        
        # Fill in the simulation results for each non-TOTAL material.
        non_total_df = non_total_df.reset_index(drop=True)
        for r in range(len(non_total_df)):
            material_name = non_total_df.loc[r, 'Material type']
            k = material_to_index[material_name]
            material_inventory_array[i, j, k, :] = sim_values[r, :]
    
    # For the TOTAL row, store the simulated total directly.
    if not total_row.empty:
        k = material_to_index['TOTAL']
        material_inventory_array[i, j, k, :] = simulated_totals

# ----------------------------
# 3. Example Query and Output
# ----------------------------
print("Unique types:", unique_types)
print("Unique cohorts:", unique_cohorts)
print("Unique materials:", unique_materials)
print("-" * 60)
print("Material inventory array shape:", material_inventory_array.shape)

# For example, retrieve simulation values for the first type, first cohort, and material "concrete" (if present).
material_name_query = "concrete" 
if material_name_query in material_to_index:
    idx_material = material_to_index[material_name_query]
    sim_vals = material_inventory_array[0, 0, idx_material, :]
    print(f"Simulated values for {unique_types[0]} / {unique_cohorts[0]} / {material_name_query}:")
    print(sim_vals)
else:
    print(f"Material '{material_name_query}' not found in the data.")


DataFrame after splitting 'Archetype' into 'type' and 'cohort':
    Archetype             Material type        average type cohort
0     AB_1955                    cement  124669.718274   AB   1955
1     AB_1955                  concrete  464048.503418   AB   1955
2     AB_1955        concrete surrogate  332089.134720   AB   1955
3     AB_1955  construction grade steel   22167.514520   AB   1955
4     AB_1955                     glass    2697.149572   AB   1955
..        ...                       ...            ...  ...    ...
242  MFH_2021                insulation    5886.716623  MFH   2021
244  MFH_2021    wood and wood products   70966.022483  MFH   2021
245  MFH_2021            wood surrogate    4614.782720  MFH   2021
246  MFH_2021       paper and cardboard   11872.785156  MFH   2021
247  MFH_2021                     TOTAL  207026.687167  MFH   2021

[224 rows x 5 columns]
------------------------------------------------------------
Unique types: ['AB' 'MFH' 'SFH']
Unique cohorts

### 2. Average heated floor area per dwelling

In [27]:
num_simulations = 500

# Each record will be: (type, cohort, sim_no, sim_value)
final_records = []

# Loop over each column in heated_floor_df (except "Combined")
for col in heated_floor_df.columns:
    if col == "Combined":
        continue

    base_val = heated_floor_df[col].iloc[0]
    
    # Clean the column name in case there is a trailing underscore.
    col_clean = col.rstrip('_')
    parts = col_clean.split('_', 1)
    if len(parts) == 2:
        typ, cohort = parts[0], parts[1]
    else:
        typ = col_clean
        cohort = ''
    
    # Set triangular distribution parameters based on cohort.
    if cohort == '1955':
        lower = base_val * 0.7
        mode  = base_val
        upper = base_val * 1.3
    elif cohort == '1956_1970':
        lower = base_val * 0.8
        mode  = base_val
        upper = base_val * 1.2
    else:
        lower = base_val * 0.9
        mode  = base_val
        upper = base_val * 1.1
        
    
    sim_values = np.random.triangular(lower, mode, upper, num_simulations)
    
    for sim_no, sim_val in enumerate(sim_values, start=1):
        final_records.append((typ, cohort, sim_no, sim_val))

# Define the structured array dtype.
final_dtype = [('type', 'U10'),
               ('cohort', 'U30'),
               ('sim_no', 'i4'),
               ('sim_value', 'f8')]

# Convert the list of records to a structured NumPy array.
HFA_array = np.array(final_records, dtype=final_dtype)

print("Final simulated array (structure: {type, cohort, sim_no, sim_value}):")
print(HFA_array)

Final simulated array (structure: {type, cohort, sim_no, sim_value}):
[('AB', '1955',  1,  67.0025921 ) ('AB', '1955',  2,  67.70035326)
 ('AB', '1955',  3,  65.8676592 ) ('AB', '1955',  4,  53.65500547)
 ('AB', '1955',  5,  42.15380279) ('AB', '1955',  6,  66.43896724)
 ('AB', '1955',  7,  54.74674847) ('AB', '1955',  8,  68.46148784)
 ('AB', '1955',  9,  68.26835622) ('AB', '1955', 10,  63.69103614)
 ('AB', '1956_1970',  1,  50.53440563)
 ('AB', '1956_1970',  2,  51.70264272)
 ('AB', '1956_1970',  3,  57.81618057)
 ('AB', '1956_1970',  4,  50.83911802)
 ('AB', '1956_1970',  5,  48.57158084)
 ('AB', '1956_1970',  6,  53.62023947)
 ('AB', '1956_1970',  7,  59.81221712)
 ('AB', '1956_1970',  8,  55.33512347)
 ('AB', '1956_1970',  9,  53.77066361)
 ('AB', '1956_1970', 10,  47.07306128)
 ('AB', '1971_1980',  1,  61.74732196)
 ('AB', '1971_1980',  2,  66.23965561)
 ('AB', '1971_1980',  3,  58.12878497)
 ('AB', '1971_1980',  4,  61.11285483)
 ('AB', '1971_1980',  5,  64.07909021)
 ('AB', '1

### ODYM-RECC data

### 3. Manufacturing demand

In [28]:
# Data from ODYM-RECC

manufacturing_energy_int = pd.read_excel("/Users/dolgayamaria/Thesis/Practical Part/Data/LCA/4_EI_ManufacturingEnergyIntensity_V2.2.xlsx", sheet_name = "Values_Master")
direct_emissions = pd.read_excel("/Users/dolgayamaria/Thesis/Practical Part/Data/LCA/6_PR_DirectEmissions_V1.2.xlsx", sheet_name = "Values_Master")
process_extentions_materials = pd.read_excel("/Users/dolgayamaria/Thesis/Practical Part/Data/CURRENT_VN1_0_from_Lola/4_PE_ProcessExtensions_Materials_VN1.0.xlsx", sheet_name = "values")

manufacturing_energy_int_copy = manufacturing_energy_int.copy()
manufacturing_energy_int_copy = manufacturing_energy_int_copy[['Manufacturing_i3', 'Energy_Carriers_m6', 'value', 'unit']]
manufacturing_energy_int_copy = manufacturing_energy_int_copy[
    manufacturing_energy_int_copy['Manufacturing_i3'].str.contains('SFH_standard|MFH_standard|RT_standard', case=False, regex=True)
]
manufacturing_energy_int_copy = manufacturing_energy_int_copy.drop_duplicates()

In [29]:
simulation_runs = 500

results = []

for manufacturing, group in manufacturing_energy_int_copy.groupby("Manufacturing_i3"):
    # Calculate the total for this manufacturing group from the 'value' column.
    total_value = group["value"].sum()
    group = group.copy()
    group["share"] = group["value"] / total_value
    # Simulate total values using a triangular distribution (±10% around the total)
    simulated_totals = np.random.triangular(total_value * 0.9, total_value, total_value * 1.1, simulation_runs)
    
    # Determine the short type name from Manufacturing_i3:
    mfg_lower = manufacturing.lower()
    if "sfh_standard" in mfg_lower:
        mfg_type = "SFH"
    elif "mfh_standard" in mfg_lower:
        mfg_type = "MFH"
    elif "rt_standard" in mfg_lower:
        mfg_type = "AB"
    else:
        mfg_type = manufacturing
        
    for sim_no in range(simulation_runs):
        sim_total = simulated_totals[sim_no]
        # For each energy carrier, calculate simulated value from share * simulated total.
        for _, row in group.iterrows():
            simulated_value = row["share"] * sim_total
            results.append((mfg_type, row["Energy_Carriers_m6"], sim_no + 1, simulated_value))

# Convert results to a structured NumPy array
dtype = [("type", "U50"), ("energy_carrier", "U50"), ("sim_no", "i4"), ("simulated_value", "f8")]
manufacturing_demand_array = np.array(results, dtype=dtype)

manufacturing_demand_array

array([('MFH', 'diesel',  1,   5.06953434),
       ('MFH', 'electricity',  1,   0.36500647),
       ('MFH', 'gasoline',  1, 158.37225291),
       ('MFH', 'diesel',  2,   5.10996711),
       ('MFH', 'electricity',  2,   0.36791763),
       ('MFH', 'gasoline',  2, 159.63537251),
       ('MFH', 'diesel',  3,   4.97672899),
       ('MFH', 'electricity',  3,   0.35832449),
       ('MFH', 'gasoline',  3, 155.47301373),
       ('MFH', 'diesel',  4,   5.06846673),
       ('MFH', 'electricity',  4,   0.3649296 ),
       ('MFH', 'gasoline',  4, 158.33890076),
       ('MFH', 'diesel',  5,   5.04410216),
       ('MFH', 'electricity',  5,   0.36317536),
       ('MFH', 'gasoline',  5, 157.57775136),
       ('MFH', 'diesel',  6,   5.27769167),
       ('MFH', 'electricity',  6,   0.3799938 ),
       ('MFH', 'gasoline',  6, 164.87508769),
       ('MFH', 'diesel',  7,   4.65074213),
       ('MFH', 'electricity',  7,   0.33485343),
       ('MFH', 'gasoline',  7, 145.28918418),
       ('MFH', 'diesel',  8

### 4. Dwelling manufacturing Global Warming Potential

For this step, it was also necessary to prepare data on electricity, since it was not included in the raw ODYM-RECC dataset. Specific values were taken from Scarlat et al. (2022) and converted into the required units for integration with the rest of the analysis.

In [30]:
def convert_gco2_kwh_to_kgco2_mj(value_gco2_per_kwh):
    value_kgco2_per_kwh = value_gco2_per_kwh / 1000.0
    
    # 1 kWh = 3.6 MJ
    value_kgco2_per_mj = value_kgco2_per_kwh / 3.6
    
    return value_kgco2_per_mj


gco2_per_kwh = 31 # For Norway from Scarlat et al. 2022
kgco2_per_mj = convert_gco2_kwh_to_kgco2_mj(gco2_per_kwh)

direct_emissions_reduced = direct_emissions[['energy carrier', 'value', 'unit']].copy()

# Replace the value for the electricity energy carrier with kgco2_per_mj
direct_emissions_reduced.loc[
    direct_emissions_reduced["energy carrier"].str.lower() == "electricity", "value"
] = kgco2_per_mj



num_simulations = 500
np.random.seed(42)

results = []

for idx, row in direct_emissions_reduced.iterrows():
    base_val = row["value"]
    lower = base_val * 0.9
    mode = base_val
    upper = base_val * 1.1

    if np.isclose(lower, mode, rtol=1e-5) or base_val == 0:
        simulated_values = np.full(num_simulations, mode)
    else:
        simulated_values = np.random.triangular(lower, mode, upper, num_simulations)
    
    for sim_no, sim_value in enumerate(simulated_values, start=1):
        results.append((row["energy carrier"], sim_no, sim_value))

# Convert results to a structured NumPy array
dtype = [("energy_carrier", "U50"), ("sim_no", "i4"), ("simulated_value", "f8")]
carriers_emissions_array = np.array(results, dtype=dtype)

carriers_emissions_array


array([('electricity',  1, 0.00849529), ('electricity',  2, 0.00920187),
       ('electricity',  3, 0.00884178), ('electricity',  4, 0.00870073),
       ('electricity',  5, 0.00823102), ('electricity',  6, 0.00823098),
       ('electricity',  7, 0.0080435 ), ('electricity',  8, 0.00902673),
       ('electricity',  9, 0.0087031 ), ('electricity', 10, 0.00881424),
       ('heat',  1, 0.        ), ('heat',  2, 0.        ),
       ('heat',  3, 0.        ), ('heat',  4, 0.        ),
       ('heat',  5, 0.        ), ('heat',  6, 0.        ),
       ('heat',  7, 0.        ), ('heat',  8, 0.        ),
       ('heat',  9, 0.        ), ('heat', 10, 0.        ),
       ('coal, hard coal',  1, 0.10123192),
       ('coal, hard coal',  2, 0.11830152),
       ('coal, hard coal',  3, 0.1146322 ),
       ('coal, hard coal',  4, 0.10616841),
       ('coal, hard coal',  5, 0.10563337),
       ('coal, hard coal',  6, 0.10566212),
       ('coal, hard coal',  7, 0.1075806 ),
       ('coal, hard coal',  8, 0

### 5. Material production Global Warming Potential

In [32]:
df = process_extentions_materials.copy()
df = df[['Material_Production_i2', 'Env_midpoints', 2022, 'unit']]
df = df[df['Env_midpoints'] == 'GWP100'].copy()

allowed_materials = material_inventory_data["Material type"].unique().tolist()
allowed_materials = [x.strip() for x in allowed_materials]
allowed_materials = sorted(allowed_materials, key=lambda x: len(x), reverse=True)

def standardize_material(mat, allowed_materials):
    mat_lower = mat.lower().strip()
    for allowed in allowed_materials:
        if allowed.lower() in mat_lower:
            return allowed
    print(f"Material type '{mat}' not recognized. Skipping.")
    return np.nan

df['Material_Production_i2'] = df['Material_Production_i2'].apply(lambda x: standardize_material(x, allowed_materials))
df = df.dropna(subset=['Material_Production_i2'])

# Simulate the 2022 values with a triangular distribution ±10%
num_simulations = 500
results = []
np.random.seed(42)

for idx, row in df.iterrows():
    base_val = row[2022]
    lower = base_val * 0.9
    mode = base_val
    upper = base_val * 1.1
    if np.isclose(lower, mode, rtol=1e-5):
        sim_values = np.full(num_simulations, mode)
    else:
        sim_values = np.random.triangular(lower, mode, upper, num_simulations)
    mat_type = row['Material_Production_i2']
    for sim_no, sim_val in enumerate(sim_values, start=1):
        results.append((mat_type, sim_no, sim_val))

dtype = [('material', 'U50'), ('sim_no', 'i4'), ('simulated_GWP', 'f8')]
GWP_material_production_array = np.array(results, dtype=dtype)

Material type 'production of copper electric grade, primary' not recognized. Skipping.
Material type 'production of automotive steel, primary' not recognized. Skipping.
Material type 'production of cast iron, primary' not recognized. Skipping.
Material type 'production of stainless steel, primary' not recognized. Skipping.
Material type 'production of wrought Al, primary' not recognized. Skipping.
Material type 'production of cast Al, primary' not recognized. Skipping.
Material type 'production of zinc, primary' not recognized. Skipping.
Material type 'production of plastics, primary' not recognized. Skipping.
Material type 'production of bricks from clay' not recognized. Skipping.
Material type 'production of bitumen' not recognized. Skipping.


### 6. Clean the datasets that will be used for the material stock and GHG emissions assessment

In [33]:
# Rename columns: replace "MFH" with "AB" and "TH" with "MFH"
dwellings_prepared.rename(columns=lambda col: col.replace("MFH", "AB").replace("TH", "MFH"), inplace=True)

# Use MunicipalitiesConverter to clean kommunenum in dwellings_prepared
municipality_converter = MunicipalitiesConverter(dwellings_prepared)
dwellings_prepared_cleaned = municipality_converter.clean_municipalities()



df = dwellings_prepared_cleaned.reset_index()
# Pivot (melt) the dataframe from wide to long format.
melted_dwellings = df.melt(id_vars='kommunenum', var_name='archetype', value_name='count')

# Split the 'archetype_year' column into 'archetype' and 'year'.
melted_dwellings[['type', 'cohort']] = melted_dwellings['archetype'].str.split('_', n=1, expand=True)

# Optionally, drop the intermediate 'archetype_year' column
melted_dwellings.drop(columns=['archetype'], inplace=True)

## Repeat the material stock assessment based on Approach 2 and perform the GHG emissions assessment both using the datasets resulted from Monte Carlo simulations

### 1. Compute total heated area across municipalities using HFA_array and melted_dwellings

In [34]:
# Convert HFA_array to a DataFrame.
hfa_df = pd.DataFrame(HFA_array)

# Merge melted_dwellings with hfa_df on 'type' and 'cohort'
merged_df = pd.merge(melted_dwellings, hfa_df, on=['type', 'cohort'])

# Multiply the 'count' (number of dwellings) with the simulated HFA value to get total HFA
merged_df['total_hfa'] = merged_df['count'] * merged_df['sim_value']

# Select desired columns: type, cohort, kommunenum, sim_no, and the computed total_hfa.
HFA_total_df = merged_df[['type', 'cohort', 'kommunenum', 'sim_no', 'total_hfa']]

# Alternatively, convert the result to NumPy array:
dtype_final = [('type', 'U10'), ('cohort', 'U30'), ('kommunenum', 'U10'), ('sim_no', 'i4'), ('total_hfa', 'f8')]
HFA_total_array = np.array(HFA_total_df.to_records(index=False), dtype=dtype_final)

### 2. Compute material intensity using material_inventory_array and archetype_area_df

In [35]:
num_simulations = 500  

# ----------------------------
# 1. Build the Average Area Structured Array using DB_Area directly
# ----------------------------
# Extract the DB_Area values as a NumPy array:
base_vals = archetype_area_df['DB_Area'].values 
sim_values = np.tile(base_vals, (num_simulations, 1)).T  

# Process the Archetype field:
# Remove any trailing underscores and split Archetype into type and cohort.
archetypes_clean = archetype_area_df['Archetype'].str.rstrip('_')
split_df = archetypes_clean.str.split('_', n=1, expand=True)
types   = split_df[0].values      
cohorts = split_df[1].values      

# Now build the structured array.
n = len(archetype_area_df)  

final_types   = np.repeat(types, num_simulations)      
final_cohorts = np.repeat(cohorts, num_simulations)     
sim_nos = np.tile(np.arange(1, num_simulations + 1), n) 
final_sim_vals = sim_values.flatten()                 

# Define the structured array dtype.
av_dtype = [
    ('type', 'U10'),
    ('cohort', 'U30'),
    ('sim_no', 'i4'),
    ('sim_value', 'f8')
]

# Allocate an empty structured array and assign values.
av_areas_array = np.empty(final_types.shape[0], dtype=av_dtype)
av_areas_array['type'] = final_types
av_areas_array['cohort'] = final_cohorts
av_areas_array['sim_no'] = sim_nos
av_areas_array['sim_value'] = final_sim_vals


# =============================================================================
# 2. Convert av_areas_array into a 3-D Array and Compute Material Intensity
# =============================================================================
# Convert the structured array to a DataFrame for convenience.
av_df = pd.DataFrame(av_areas_array)
unique_types_av = np.sort(av_df['type'].unique())
unique_cohorts_av = np.sort(av_df['cohort'].unique())

n_types_av = len(unique_types_av)
n_cohorts_av = len(unique_cohorts_av)

# Allocate a new 3-D array with shape (n_types, n_cohorts, num_simulations)
av_areas_3d = np.empty((n_types_av, n_cohorts_av, num_simulations), dtype=float)

# Fill the 3-D array with sim_value, matching by type and cohort
for rec in av_areas_array:
    i = np.where(unique_types_av == rec['type'])[0][0]
    j = np.where(unique_cohorts_av == rec['cohort'])[0][0]
    sim_index = rec['sim_no'] - 1  
    av_areas_3d[i, j, sim_index] = rec['sim_value']


# ------------------------------------------------------------------------------
# 3. Compute Material Intensity by Broadcasted Division
# ------------------------------------------------------------------------------
material_intensity_array = material_inventory_array / av_areas_3d[:, :, None, :]

# ------------------------------------------------------------------------------
# 4. Build a Tidy DataFrame with Material Intensity Values
# ------------------------------------------------------------------------------
n_types, n_cohorts, n_materials, _ = material_inventory_array.shape

records = []
for i, typ in enumerate(unique_types_av):
    for j, coh in enumerate(unique_cohorts_av):
        for k in range(n_materials):
            for sim_no in range(num_simulations):
                intensity = material_intensity_array[i, j, k, sim_no]
                material_name = unique_materials[k]  # unique_materials should exist
                records.append((typ, coh, material_name, sim_no + 1, intensity))

df_columns = ['type', 'cohort', 'material', 'sim_no', 'material_intensity']
material_intensity_df = pd.DataFrame(records, columns=df_columns)

### 3. Compute total material stock using material_intensity_df and hfa_total_df

In [36]:
# ----------------------------
# 1. Merge the DataFrames
# ----------------------------
merged_df = pd.merge(material_intensity_df, HFA_total_df, on=['type', 'cohort', 'sim_no'], how='inner')

# ----------------------------
# 2. Compute the Material Stock
# ----------------------------
# The material stock is computed as:
#     material_stock = material_intensity * total_hfa
merged_df['material_stock'] = merged_df['material_intensity'] * merged_df['total_hfa']

# ----------------------------
# 3. Select the Desired Columns 
# ----------------------------
material_stock_df = merged_df[['type', 'cohort', 'material', 'kommunenum', 'sim_no', 'material_stock']]

# ----------------------------
# 4. (Optional) Convert to a NumPy Structured Array
# ----------------------------
final_dtype = [
    ('type', 'U10'),
    ('cohort', 'U30'),
    ('material', 'U30'),
    ('kommunenum', 'U10'),
    ('sim_no', 'i4'),
    ('material_stock', 'f8')
]

material_stock_array = np.array(material_stock_df.to_records(index=False), dtype=final_dtype)


# aggregate material stocks 
# Group by the desired columns and compute the mean and std of material_stock
agg_stats = material_stock_df.groupby(
    ['type', 'cohort', 'material', 'kommunenum'], as_index=False
).agg(
    material_stock_mean=('material_stock', 'mean'),
    material_stock_min=('material_stock', 'min'),
    material_stock_max=('material_stock', 'max'),
    material_stock_std=('material_stock', 'std'),
    material_stock_p5 = ('material_stock', lambda x: np.percentile(x, 5)),
    material_stock_p95 = ('material_stock', lambda x: np.percentile(x, 95))
)

# Compute the standard deviation percentage relative to the mean.
agg_stats['material_stock_std_pct'] = (agg_stats['material_stock_std'] / agg_stats['material_stock_mean']) * 100

# Optionally, drop the raw standard deviation if you only need the mean and percentage.
material_stock_agg = agg_stats.drop(columns=['material_stock_std'])
material_stock_agg = material_stock_agg.fillna(0)

### 4. Compute GWP from material production by using GWP_material_production_array and material_stock_df

In [37]:
# ------------------------------------------------------------------------------
# 1. Convert GWP_material_production_array to a DataFrame
# ------------------------------------------------------------------------------
gwp_df = pd.DataFrame(GWP_material_production_array)
# gwp_df has columns: 'material', 'sim_no', 'simulated_GWP'

# ------------------------------------------------------------------------------
# 2. Augment gwp_df to include surrogate materials if missing
# ------------------------------------------------------------------------------
# For 'wood surrogate': if not present, duplicate rows where material == 'wood and wood products'
if not (gwp_df['material'].str.lower() == 'wood surrogate').any():
    wood_rows = gwp_df[gwp_df['material'].str.lower() == 'wood and wood products']
    if not wood_rows.empty:
        surrogate_wood = wood_rows.copy()
        surrogate_wood['material'] = 'wood surrogate'
        gwp_df = pd.concat([gwp_df, surrogate_wood], ignore_index=True)

# For 'concrete surrogate': if not present, duplicate rows where material == 'concrete'
if not (gwp_df['material'].str.lower() == 'concrete surrogate').any():
    concrete_rows = gwp_df[gwp_df['material'].str.lower() == 'concrete']
    if not concrete_rows.empty:
        surrogate_concrete = concrete_rows.copy()
        surrogate_concrete['material'] = 'concrete surrogate'
        gwp_df = pd.concat([gwp_df, surrogate_concrete], ignore_index=True)

# ------------------------------------------------------------------------------
# 3. Merge material_stock_df with gwp_df on 'material' and 'sim_no'
# ------------------------------------------------------------------------------
merged_df = pd.merge(material_stock_df, gwp_df, on=['material', 'sim_no'], how='left')

# ------------------------------------------------------------------------------
# 4. Compute the Emissions per Material
# ------------------------------------------------------------------------------
# For rows where GWP data exists, compute:
merged_df['material_emissions'] = merged_df['material_stock'] * merged_df['simulated_GWP']

# ------------------------------------------------------------------------------
# 5. Compute Aggregate (TOTAL) Emissions per Group (per simulation run)
# ------------------------------------------------------------------------------
# For each combination of type, cohort, kommunenum, and sim_no, sum the individual emissions.
group_cols = ['type', 'cohort', 'kommunenum', 'sim_no']
agg_total = merged_df.groupby(group_cols)['material_emissions'].sum().reset_index()
agg_total['material'] = 'TOTAL'

# ------------------------------------------------------------------------------
# 6. Build the Final Dataset Without Removing Individual Rows
# ------------------------------------------------------------------------------
# Remove any existing rows for material == "TOTAL" from merged_df (if any) to avoid duplication.
indiv_df = merged_df[merged_df['material'] != 'TOTAL'].copy()

# Concatenate the individual rows with the aggregated TOTAL rows.
material_stock_emissions_df = pd.concat([indiv_df, agg_total], ignore_index=True)

# Arrange the final columns in a logical order.
material_stock_emissions_df = material_stock_emissions_df[
    ['type', 'cohort', 'material', 'kommunenum', 'sim_no', 'material_emissions']
]

# At this point, material_stock_emissions_df has one row per simulation run (sim_no) for each
# combination of type, cohort, material, and kommunenum. For each simulation run (sim_no),
# there is a row for an individual material (e.g. wood, wood surrogate, concrete, etc.)
# and a row for TOTAL (the sum over all materials).

# ------------------------------------------------------------------------------
# 7. Aggregate Over Simulation Runs to Compute Summary Statistics
# ------------------------------------------------------------------------------
# For each unique combination of type, cohort, material, and kommunenum (ignoring sim_no),
# we now want to store aggregated statistics over the 10 simulation runs.
ms_emissions_agg = material_stock_emissions_df.groupby(
    ['type', 'cohort', 'material', 'kommunenum'], as_index=False
).agg(
    material_emissions_mean=('material_emissions', 'mean'),
    material_emissions_min=('material_emissions', 'min'),
    material_emissions_max=('material_emissions', 'max'),
    material_emissions_std=('material_emissions', 'std'),
    material_emissions_p5 = ('material_emissions', lambda x: np.percentile(x, 5)),
    material_emissions_p95 = ('material_emissions', lambda x: np.percentile(x, 95))
)

ms_emissions_agg['material_emissions_std_pct'] = (
    ms_emissions_agg['material_emissions_std'] / ms_emissions_agg['material_emissions_mean']
) * 100


ms_emissions_agg = ms_emissions_agg.fillna(0)
# ------------------------------------------------------------------------------
# 8. (Optional) Convert the Aggregated DataFrame to a NumPy Structured Array
# ------------------------------------------------------------------------------
final_dtype = [
    ('type', 'U10'),
    ('cohort', 'U30'),
    ('material', 'U30'),
    ('kommunenum', 'U10'),
    ('material_emissions_mean', 'f8'),
    ('material_emissions_min', 'f8'),
    ('material_emissions_max', 'f8'),
    ('material_emissions_std', 'f8'),
    ('material_emissions_std_pct', 'f8'),
    ('material_emissions_p5', 'f8'),
    ('material_emissions_p95', 'f8')
]
material_emissions_stats_array = np.array(ms_emissions_agg.to_records(index=False), dtype=final_dtype)

### 5. Compute GWP from manufacturing using manufacturing_demand_array and carriers_emissions_array

In [38]:
# Convert manufacturing_demand_array to DataFrame and rename simulated_value to demand_value.
manufacturing_df = pd.DataFrame(manufacturing_demand_array)
manufacturing_df = manufacturing_df.rename(columns={'simulated_value': 'demand_value'})
# manufacturing_df now has columns: type, energy_carrier, sim_no, demand_value

# Convert carriers_emissions_array to DataFrame and rename simulated_value to emission_factor.
carriers_df = pd.DataFrame(carriers_emissions_array)
carriers_df = carriers_df.rename(columns={'simulated_value': 'emission_factor'})
# carriers_df now has columns: energy_carrier, sim_no, emission_factor

# Merge the two DataFrames on 'energy_carrier' and 'sim_no'
gwp_df = pd.merge(manufacturing_df, carriers_df, on=['energy_carrier', 'sim_no'], how='inner')

# Compute the GWP for each entry
gwp_df['gwp_value'] = gwp_df['demand_value'] * gwp_df['emission_factor']

# Select only the desired columns (excluding any totals for now)
GWP_manufacturing_df = gwp_df[['type', 'energy_carrier', 'sim_no', 'gwp_value']]

# Optionally, convert the DataFrame to a structured NumPy array.
final_dtype = [
    ('type', 'U50'),
    ('energy_carrier', 'U50'),
    ('sim_no', 'i4'),
    ('gwp_value', 'f8')
]
GWP_manufacturing_array = np.array(GWP_manufacturing_df.to_records(index=False), dtype=final_dtype)

### 6. Compute manufacturing emissions using GWP_manufacturing_df and HFA_total_df

In [39]:
merged_manufacturing = pd.merge(HFA_total_df, GWP_manufacturing_df, on=['type', 'sim_no'], how='inner')

# The merged DataFrame now has columns:
#   type, cohort, kommunenum, sim_no, total_hfa, energy_carrier, gwp_value

# Step 2: Compute the manufacturing emissions per row.
# Multiply the total heated area by the emission factor:
merged_manufacturing['manufacturing_emissions'] = merged_manufacturing['total_hfa'] * merged_manufacturing['gwp_value']

# Select only the desired columns:
manufacturing_emissions_df = merged_manufacturing[['type', 'cohort', 'kommunenum', 'energy_carrier', 'sim_no', 'manufacturing_emissions']]


# Step 3: Compute Aggregated TOTAL Emissions per Simulation Run.
# For each unique combination of type, cohort, kommunenum, and sim_no,
# sum the manufacturing_emissions over all energy carriers.
total_emissions = manufacturing_emissions_df.groupby(
    ['type', 'cohort', 'kommunenum', 'sim_no'], as_index=False
)['manufacturing_emissions'].sum()

# Mark these as TOTAL rows.
total_emissions['energy_carrier'] = 'TOTAL'
total_emissions = total_emissions[['type', 'cohort', 'kommunenum', 'energy_carrier', 'sim_no', 'manufacturing_emissions']]

# Step 4: Append the TOTAL rows to the individual energy carrier rows.
final_manufacturing_emissions_df = pd.concat([manufacturing_emissions_df, total_emissions], ignore_index=True)

# (Optional) Sort the final DataFrame for clarity.
final_manufacturing_emissions_df = final_manufacturing_emissions_df.sort_values(
    by=['type', 'cohort', 'kommunenum', 'sim_no', 'energy_carrier']
).reset_index(drop=True)

# ==============================================================================
# Now, Aggregate Over the Simulation Runs
# ==============================================================================
# For each unique combination of type, cohort, kommunenum, and energy_carrier (including TOTAL),
# compute summary statistics across all simulation runs:
#   - manufacturing_emissions_mean
#   - manufacturing_emissions_min
#   - manufacturing_emissions_max
#   - manufacturing_emissions_p5   (5th percentile)
#   - manufacturing_emissions_p95  (95th percentile)
manufacturing_emissions_agg = final_manufacturing_emissions_df.groupby(
    ['type', 'cohort', 'kommunenum', 'energy_carrier'], as_index=False
).agg(
    manufacturing_emissions_mean = ('manufacturing_emissions', 'mean'),
    manufacturing_emissions_min  = ('manufacturing_emissions', 'min'),
    manufacturing_emissions_max  = ('manufacturing_emissions', 'max'),
    manufacturing_emissions_std  = ('manufacturing_emissions', 'std'),
    manufacturing_emissions_p5   = ('manufacturing_emissions', lambda x: np.percentile(x, 5)),
    manufacturing_emissions_p95  = ('manufacturing_emissions', lambda x: np.percentile(x, 95))
)

manufacturing_emissions_agg['manufacturing_emissions_std_pct'] = (
    manufacturing_emissions_agg['manufacturing_emissions_std'] / manufacturing_emissions_agg['manufacturing_emissions_mean']
) * 100

manufacturing_emissions_agg = manufacturing_emissions_agg.fillna(0)
# ==============================================================================
# (Optional) Convert the Aggregated DataFrame to a NumPy Structured Array
# ==============================================================================
final_dtype = [
    ('type', 'U50'),
    ('cohort', 'U30'),
    ('kommunenum', 'U10'),
    ('energy_carrier', 'U50'),
    ('manufacturing_emissions_mean', 'f8'),
    ('manufacturing_emissions_min', 'f8'),
    ('manufacturing_emissions_max', 'f8'),
    ('manufacturing_emissions_std', 'f8'),
    ('manufacturing_emissions_std_pct', 'f8'),
    ('manufacturing_emissions_p5', 'f8'),
    ('manufacturing_emissions_p95', 'f8')
]

### 7. Merge the results 

In [43]:
# ------------------------------------------------------------------------------
# 1. Filter for TOTAL rows in each dataset
# ------------------------------------------------------------------------------
# For manufacturing, TOTAL rows are those where energy_carrier == "TOTAL"
manufacturing_TOTAL = final_manufacturing_emissions_df[
    final_manufacturing_emissions_df['energy_carrier'] == 'TOTAL'
].copy()

# For materials, TOTAL rows are those where material == "TOTAL"
materials_TOTAL = material_stock_emissions_df[
    material_stock_emissions_df['material'] == 'TOTAL'
].copy()

# ------------------------------------------------------------------------------
# 2. Merge the filtered datasets on the common keys: type, cohort, kommunenum, sim_no
# ------------------------------------------------------------------------------
merged_emissions_df = pd.merge(
    manufacturing_TOTAL,
    materials_TOTAL,
    on=['type', 'cohort', 'kommunenum', 'sim_no'],
    how='outer'
)

# ------------------------------------------------------------------------------
# 3. Fill missing emission columns with 0
# ------------------------------------------------------------------------------
merged_emissions_df['manufacturing_emissions'] = merged_emissions_df['manufacturing_emissions'].fillna(0)
merged_emissions_df['material_emissions'] = merged_emissions_df['material_emissions'].fillna(0)

# ------------------------------------------------------------------------------
# 4. Create a new column "total_emissions" that is the sum of manufacturing and material emissions
# ------------------------------------------------------------------------------
merged_emissions_df['total_emissions'] = (
    merged_emissions_df['manufacturing_emissions'] + merged_emissions_df['material_emissions']
)

# For clarity, rearrange columns:
final_cols = ['type', 'cohort', 'kommunenum', 'sim_no', 
              'energy_carrier', 'material', 'total_emissions']
merged_emissions_df = merged_emissions_df[final_cols]


# ------------------------------------------------------------------------------
# 5. Aggregate across all simulations to compute summary statistics:
#    total_mean and total_std (as percentage of the mean)
# ------------------------------------------------------------------------------
# Group by type, cohort, and kommunenum to aggregate across simulation runs
summary_df = merged_emissions_df.groupby(['type', 'cohort', 'kommunenum'], as_index=False).agg(
    total_mean = ('total_emissions', 'mean'),
    total_std = ('total_emissions', 'std')
)

# Compute the standard deviation as a percentage of the mean:
summary_df['total_std_pct'] = (summary_df['total_std'] / summary_df['total_mean']) * 100

# Rearrange the output columns:
summary_df = summary_df[['type', 'cohort', 'kommunenum', 'total_mean', 'total_std_pct']]

# =============================================================================

# ------------------------------------------------------------------------------
# 6. (Optional) Convert the Summary DataFrame to a NumPy Structured Array
# ------------------------------------------------------------------------------
final_dtype = [
    ('type', 'U50'),
    ('cohort', 'U30'),
    ('kommunenum', 'U10'),
    ('total_mean', 'f8'),
    ('total_std_pct', 'f8')
]
summary_array = np.array(summary_df.to_records(index=False), dtype=final_dtype)
summary_df = summary_df.fillna(0)
summary_df

,type,cohort,kommunenum,total_mean,total_std_pct
0,AB,1955,0301,1.862535e+09,24.171724
1,AB,1955,1101,1.102763e+05,24.171724
2,AB,1955,1103,2.848279e+07,24.171724
3,AB,1955,1106,7.104943e+06,24.171724
4,AB,1955,1108,3.544595e+06,24.171724
...,...,...,...,...,...
8563,SFH,2021,5628,4.102723e+05,7.384822
8564,SFH,2021,5630,0.000000e+00,0.000000
8565,SFH,2021,5632,0.000000e+00,0.000000
8566,SFH,2021,5634,5.861033e+04,7.384822


In [ ]:
# Export the results 

summary_df.to_csv("summary_df.csv", index=False)
material_stock_agg.to_csv("material_stock_agg.csv", index=False)
ms_emissions_agg.to_csv("ms_emissions_agg.csv", index=False)
manufacturing_emissions_agg.to_csv("manufacturing_emissions_agg.csv", index=False)

In [47]:
total_heated_area_melted.to_csv("total_heated_area_melted.csv", index=False)

### 8. Merge results for the dashboard

In [42]:
ms_ms_emissions_merged = pd.merge(
    material_stock_agg,
    ms_emissions_agg,
    on=["kommunenum", "type", "cohort", "material"],
    how="outer"  
)

merged_dashboard_df = pd.merge(
    ms_ms_emissions_merged,
    manufacturing_emissions_agg,
    on=["kommunenum", "type", "cohort"],
    how="outer"  
)
merged_dashboard_df

,type,cohort,material,kommunenum,material_stock_mean,material_stock_min,material_stock_max,material_stock_p5,material_stock_p95,material_stock_std_pct,...,material_emissions_p95,material_emissions_std_pct,energy_carrier,manufacturing_emissions_mean,manufacturing_emissions_min,manufacturing_emissions_max,manufacturing_emissions_std,manufacturing_emissions_p5,manufacturing_emissions_p95,manufacturing_emissions_std_pct
0,AB,1955,TOTAL,0301,1.062993e+10,6.194473e+09,1.382700e+10,6.754596e+09,1.357857e+10,23.053627,...,2.331874e+09,24.852258,TOTAL,8.176419e+07,5.733348e+07,9.647938e+07,1.157896e+07,6.373748e+07,9.447110e+07,14.161403
1,AB,1955,TOTAL,0301,1.062993e+10,6.194473e+09,1.382700e+10,6.754596e+09,1.357857e+10,23.053627,...,2.331874e+09,24.852258,diesel,2.594821e+06,1.755804e+06,2.967900e+06,4.081581e+05,1.934472e+06,2.951226e+06,15.729720
2,AB,1955,TOTAL,0301,1.062993e+10,6.194473e+09,1.382700e+10,6.754596e+09,1.357857e+10,23.053627,...,2.331874e+09,24.852258,electricity,2.285930e+04,1.471309e+04,2.724464e+04,4.043592e+03,1.660008e+04,2.706942e+04,17.689045
3,AB,1955,TOTAL,0301,1.062993e+10,6.194473e+09,1.382700e+10,6.754596e+09,1.357857e+10,23.053627,...,2.331874e+09,24.852258,gasoline,7.914651e+07,5.556296e+07,9.362145e+07,1.120544e+07,6.174263e+07,9.155283e+07,14.157846
4,AB,1955,cement,0301,1.269420e+09,7.397403e+08,1.651213e+09,8.066298e+08,1.621545e+09,23.053627,...,1.460538e+09,26.554655,TOTAL,8.176419e+07,5.733348e+07,9.647938e+07,1.157896e+07,6.373748e+07,9.447110e+07,14.161403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371275,SFH,2021,wood surrogate,5636,3.560450e+03,3.211139e+03,3.947926e+03,3.259210e+03,3.892075e+03,6.189580,...,4.860328e+02,8.936291,TOTAL,4.147045e+03,3.568209e+03,4.585518e+03,3.227901e+02,3.701154e+03,4.563564e+03,7.783615
371276,SFH,2021,wood surrogate,5636,3.560450e+03,3.211139e+03,3.947926e+03,3.259210e+03,3.892075e+03,6.189580,...,4.860328e+02,8.936291,diesel,1.394448e+02,1.311624e+02,1.575451e+02,8.132868e+00,1.322910e+02,1.538286e+02,5.832322
371277,SFH,2021,wood surrogate,5636,3.560450e+03,3.211139e+03,3.947926e+03,3.259210e+03,3.892075e+03,6.189580,...,4.860328e+02,8.936291,electricity,9.514776e+01,8.706997e+01,1.009637e+02,5.006275e+00,8.762945e+01,1.003107e+02,5.261579
371278,SFH,2021,wood surrogate,5636,3.560450e+03,3.211139e+03,3.947926e+03,3.259210e+03,3.892075e+03,6.189580,...,4.860328e+02,8.936291,gasoline,3.445578e+03,2.921245e+03,3.811042e+03,2.906572e+02,3.039233e+03,3.805917e+03,8.435657


In [ ]:
# Export the dataset 

merged_dashboard_df.to_csv("merged_df_dashboard.csv", index = False)

In [2]:
summary_df = pd.read_csv('summary_df.csv',dtype={"kommunenum": str})

In [3]:
summary_df

,type,cohort,kommunenum,total_mean,total_std_pct
0,AB,1955,0301,1.668329e+09,18.939960
1,AB,1955,1101,9.877780e+04,18.939960
2,AB,1955,1103,2.551289e+07,18.939960
3,AB,1955,1106,6.364112e+06,18.939960
4,AB,1955,1108,3.175001e+06,18.939960
...,...,...,...,...,...
8563,SFH,2021,5628,4.051787e+05,6.040434
8564,SFH,2021,5630,0.000000e+00,0.000000
8565,SFH,2021,5632,0.000000e+00,0.000000
8566,SFH,2021,5634,5.788268e+04,6.040434


In [6]:
filtered = summary_df[summary_df['cohort'] == '2021']

# sum total_mean
total_sum = filtered['total_mean'].sum()

# divide by 4, then convert to megatonnes
value_megatonnes = (total_sum / 4) / 1_000_000_000

print(f"Result: {value_megatonnes:.4f} Mt")

Result: 0.3219 Mt


In [7]:
0.3219/10.6

0.03036792452830189